# P3 Kaggle Notebook — E0, Overfit, Kill-Resume, T7, E2

**Architecture refs:** ARCHITECTURE.md C, E1–E5, G(P3), E8.

Every section writes a JSON to `/kaggle/working/results/`.  
No corpus text is ever printed or stored in cell outputs.

## 0. Parameters

In [ ]:
import os
import sys
from pathlib import Path

# ── Paths (adjust dataset mount names if different) ──────────────────────
KAGGLE_INPUT     = Path('/kaggle/input')
KAGGLE_WORKING   = Path('/kaggle/working')
RESULTS_DIR      = KAGGLE_WORKING / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Dataset mount names (adjust to match your Kaggle dataset slugs)
CODE_DATASET     = KAGGLE_INPUT / 'capstone-tokenizer-sft-code'
MODEL_DATASET    = KAGGLE_INPUT / 'capstone-model-weights'
DATA_DATASET     = KAGGLE_INPUT / 'capstone-train-data'

PKG_DIR          = CODE_DATASET / 'tokenizer_pkg'
MODEL_DIR        = MODEL_DATASET / 'model_snapshot'
PILOT_JSONL      = DATA_DATASET / 'pilot_mix.jsonl'
OVERFIT_JSONL    = DATA_DATASET / 'overfit_set.jsonl'
VAL_MD_JSONL     = DATA_DATASET / 'val_meddialog_tok.jsonl'
VAL_HC_JSONL     = DATA_DATASET / 'val_healthchat_tok.jsonl'

# Add code path
sys.path.insert(0, str(CODE_DATASET))

# CUDA_VISIBLE_DEVICES MUST be set before any torch import
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['HF_HUB_OFFLINE'] = '1'

print('Paths configured.')
print(f'PKG_DIR exists:   {PKG_DIR.exists()}')
print(f'MODEL_DIR exists: {MODEL_DIR.exists()}')
print(f'PILOT exists:     {PILOT_JSONL.exists()}')

## 1. Definitions

In [ ]:
import hashlib
import json
import subprocess
import time

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1<<20), b''): h.update(chunk)
    return h.hexdigest()

def write_result(name, data):
    path = RESULTS_DIR / f'{name}.json'
    with open(path, 'w', encoding='utf-8', newline='\n') as f:
        json.dump(data, f, ensure_ascii=False, indent=2, sort_keys=True)
        f.write('\n')
    print(f'  Written: {path}')
    return path

print('Definitions loaded.')

## 2. In-notebook tests (must pass before reading data)

In [ ]:
import unittest

class TestKaggleSetup(unittest.TestCase):

    def test_cuda_env_set(self):
        assert os.environ.get('CUDA_VISIBLE_DEVICES') == '0', \
            'CUDA_VISIBLE_DEVICES must be 0 before torch import'

    def test_hf_offline(self):
        assert os.environ.get('HF_HUB_OFFLINE') == '1', \
            'HF_HUB_OFFLINE must be 1'

    def test_paths_exist(self):
        assert PKG_DIR.exists(), f'PKG_DIR not found: {PKG_DIR}'
        assert MODEL_DIR.exists(), f'MODEL_DIR not found: {MODEL_DIR}'
        assert PILOT_JSONL.exists(), f'PILOT_JSONL not found: {PILOT_JSONL}'
        assert OVERFIT_JSONL.exists(), f'OVERFIT_JSONL not found: {OVERFIT_JSONL}'
        assert VAL_MD_JSONL.exists(), f'VAL_MD_JSONL not found: {VAL_MD_JSONL}'
        assert VAL_HC_JSONL.exists(), f'VAL_HC_JSONL not found: {VAL_HC_JSONL}'

    def test_tokenize_mask_importable(self):
        import tokenize_mask  # noqa: F401
        assert True

    def test_results_dir_writable(self):
        test_path = RESULTS_DIR / '_test_write.tmp'
        test_path.write_text('ok')
        test_path.unlink()

loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestKaggleSetup)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), f'Setup tests failed: {result.failures} {result.errors}'

## Section: ENV — Environment check and T1-T6 on Kaggle

In [ ]:
# Run kaggle_env_check.py and gpu_probe.py
import subprocess, sys

env_result = subprocess.run(
    [sys.executable, str(CODE_DATASET / 'kaggle_env_check.py'), '--json', str(RESULTS_DIR / 'kaggle_env.json')],
    capture_output=True, text=True
)
print('kaggle_env_check stdout:', env_result.stdout[-500:])
if env_result.returncode != 0:
    print('STDERR:', env_result.stderr[-500:])
    raise RuntimeError(f'kaggle_env_check failed (exit {env_result.returncode})')

gpu_result = subprocess.run(
    [sys.executable, str(CODE_DATASET / 'gpu_probe.py'), '--json', str(RESULTS_DIR / 'gpu_probe.json')],
    capture_output=True, text=True
)
print('gpu_probe stdout:', gpu_result.stdout[-500:])
if gpu_result.returncode != 0:
    print('STDERR:', gpu_result.stderr[-500:])
    raise RuntimeError(f'gpu_probe failed (exit {gpu_result.returncode})')

In [ ]:
# pip freeze for version pinning
freeze_result = subprocess.run([sys.executable, '-m', 'pip', 'freeze'],
                               capture_output=True, text=True)
freeze_path = RESULTS_DIR / 'pip_freeze.txt'
freeze_path.write_text(freeze_result.stdout, encoding='utf-8')
print(f'pip freeze: {len(freeze_result.stdout.splitlines())} packages -> {freeze_path}')

In [ ]:
# T1-T6: Run test_tokenize_mask tests on Kaggle's transformers version
import unittest, sys

# Add test file location
tests_path = CODE_DATASET / 'tests'
if str(tests_path) not in sys.path and tests_path.exists():
    sys.path.insert(0, str(CODE_DATASET))

from tests.test_tokenize_mask import (
    TestT1SegmentVsFullString,
    TestT2SupervisedContent,
    TestT3NoLeakage,
    TestT4AssistantEndsWithEos,
    TestT5TruncationEndsOnAssistant,
    TestT6TrainInferenceParity,
    TestTSpecialTokenSanitisation,
)

# Override PKG_DIR in test module
import tests.test_tokenize_mask as _tm
_tm.PKG_DIR = PKG_DIR
_tm._TOK = None  # reset cached tokenizer

suite = unittest.TestSuite()
for cls in [TestT1SegmentVsFullString, TestT2SupervisedContent,
            TestT3NoLeakage, TestT4AssistantEndsWithEos,
            TestT5TruncationEndsOnAssistant, TestT6TrainInferenceParity,
            TestTSpecialTokenSanitisation]:
    suite.addTests(unittest.TestLoader().loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)

t16_report = {
    'tests_run': test_result.testsRun,
    'failures': len(test_result.failures),
    'errors': len(test_result.errors),
    'skipped': len(test_result.skipped),
    'pass': test_result.wasSuccessful(),
    'failure_names': [str(f[0]) for f in test_result.failures],
    'error_names': [str(e[0]) for e in test_result.errors],
}
write_result('t1_t6_results', t16_report)
assert test_result.wasSuccessful(), f'T1-T6 FAILED: {t16_report}'

## Section: E0 — GPU memory and throughput check

In [ ]:
import torch
from tokenize_mask import load_tokenizer
from train_lora import check_gpu_environment, make_collator, TokenizedDataset

assert torch.cuda.device_count() == 1, \
    f'Expected 1 GPU, got {torch.cuda.device_count()}. Set CUDA_VISIBLE_DEVICES=0.'

gpu_info = check_gpu_environment()
print(f'GPU: {gpu_info["device_name"]}, VRAM: {gpu_info["vram_gb"]:.1f} GB')
print(f'Precision: {gpu_info["precision"]}')
PRECISION = gpu_info['precision']
DTYPE = torch.bfloat16 if PRECISION == 'bf16' else torch.float16

# Load tokenizer
tok = load_tokenizer(PKG_DIR)
print(f'Tokenizer loaded. pad_id={tok.pad_token_id} eos_id={tok.eos_token_id}')

In [ ]:
# Load model for E0
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

print(f'Loading model from {MODEL_DIR}...')
model_e0 = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=DTYPE,
    device_map={'': 0},
    attn_implementation='sdpa',
    local_files_only=True,
)
model_e0.config.use_cache = False

# Apply LoRA
target_modules = list({
    name.split('.')[-1]
    for name, m in model_e0.named_modules()
    if isinstance(m, torch.nn.Linear)
    and 'lm_head' not in name and 'embed_tokens' not in name
})
lora_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=target_modules, bias='none', task_type=TaskType.CAUSAL_LM
)
model_e0 = get_peft_model(model_e0, lora_cfg)
for n, p in model_e0.named_parameters():
    if 'lora_' in n:
        p.data = p.data.to(torch.float32)

trainable = sum(p.numel() for p in model_e0.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

In [ ]:
# E0: 50 steps at micro-batch 1/2/4 on max-length batches
# Measure peak VRAM, NaN/inf check, tok/s on realistic batches
import gc
from torch.optim import AdamW

MAX_LEN_E0 = 2048
N_STEPS_E0 = 50
LR_E0 = 1e-4

# Build max-length synthetic batches for VRAM measurement
def make_max_len_batch(bs, max_len, pad_id, device):
    ids = torch.full((bs, max_len), pad_id, dtype=torch.long, device=device)
    # first token non-pad to simulate real content
    ids[:, 0] = 1
    # labels: mask everything except last 10 tokens
    labels = torch.full((bs, max_len), -100, dtype=torch.long, device=device)
    labels[:, -10:] = ids[:, -10:]
    mask = (ids != pad_id).long()
    return {'input_ids': ids, 'labels': labels, 'attention_mask': mask}

device = torch.device('cuda:0')
optimizer = AdamW(model_e0.parameters(), lr=LR_E0, weight_decay=0.0)

e0_results = {}

for micro_bs in [1, 2, 4]:
    print(f'\nE0 micro_batch={micro_bs}...')
    model_e0.train()
    torch.cuda.reset_peak_memory_stats(device)
    nan_inf_count = 0
    t_start = time.time()

    for step in range(N_STEPS_E0):
        batch = make_max_len_batch(micro_bs, MAX_LEN_E0, tok.pad_token_id, device)
        outputs = model_e0(**batch)
        loss = outputs.loss

        if not torch.isfinite(loss):
            nan_inf_count += 1

        optimizer.zero_grad()
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model_e0.parameters(), 1.0)
        if not torch.isfinite(grad_norm):
            nan_inf_count += 1

        optimizer.step()

        if step == 0:
            first_loss = loss.item()

    t_elapsed = time.time() - t_start
    peak_vram = torch.cuda.max_memory_allocated(device) / (1024**3)
    total_vram = gpu_info['vram_gb']
    vram_pct = peak_vram / total_vram * 100

    e0_results[f'micro_batch_{micro_bs}'] = {
        'peak_vram_gb': round(peak_vram, 3),
        'total_vram_gb': total_vram,
        'vram_pct': round(vram_pct, 1),
        'nan_inf_count': nan_inf_count,
        'steps': N_STEPS_E0,
        'time_s': round(t_elapsed, 1),
        'fit_90pct': vram_pct < 90.0 and nan_inf_count == 0,
    }
    print(f'  peak VRAM: {peak_vram:.2f} GB / {total_vram:.1f} GB ({vram_pct:.1f}%)')
    print(f'  NaN/inf: {nan_inf_count}, fit_90pct: {e0_results[f"micro_batch_{micro_bs}"]["fit_90pct"]}')
    gc.collect()
    torch.cuda.empty_cache()

# Measure realistic tok/s on length-grouped batches from pilot mix
def load_tok_jsonl(path, n=64):
    items = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
            if len(items) >= n:
                break
    return items

print('\nMeasuring realistic tok/s...')
pilot_items = load_tok_jsonl(PILOT_JSONL, 64)
collate_fn = make_collator(tok.pad_token_id)

# Use micro_batch from best E0 result
best_mb = max(
    (mb for mb in [1,2,4] if e0_results.get(f'micro_batch_{mb}', {}).get('fit_90pct', False)),
    default=1
)
print(f'Best micro_batch from E0: {best_mb}')

# Time 10 steps on realistic batches
model_e0.train()
optimizer2 = AdamW(model_e0.parameters(), lr=LR_E0, weight_decay=0.0)
tok_count = 0
t_tok_start = time.time()
for step_i in range(0, min(len(pilot_items), 10 * best_mb), best_mb):
    batch_items = pilot_items[step_i:step_i + best_mb]
    batch = collate_fn(batch_items)
    # Count real (non-pad) tokens
    real_toks = int(batch['attention_mask'].sum())
    batch = {k: v.to(device) if hasattr(v, 'to') else v for k, v in batch.items()}
    outputs = model_e0(**{k: v for k, v in batch.items() if k != 'num_items_in_batch'})
    outputs.loss.backward()
    optimizer2.step()
    optimizer2.zero_grad()
    tok_count += real_toks
tok_elapsed = time.time() - t_tok_start
toks_per_sec = tok_count / max(tok_elapsed, 1e-6)

e0_results['realistic_toks_per_sec'] = round(toks_per_sec, 1)
e0_results['best_micro_batch'] = best_mb
e0_results['gpu'] = gpu_info
print(f'Realistic tok/s: {toks_per_sec:.1f} (micro_batch={best_mb})')

write_result('e0_results', e0_results)
del model_e0, optimizer, optimizer2
gc.collect()
torch.cuda.empty_cache()
print('E0 complete.')

## Section: Overfit — 16 dialogues × 150 steps

In [ ]:
# Overfit test: 16 dialogues, 150 steps
# Acceptance: final loss <= 30% of initial loss
MICRO_BATCH_OVERFIT = e0_results['best_micro_batch']
OVERFIT_STEPS = 150
LR_OVERFIT = 1e-3  # higher LR to accelerate overfit on tiny set

overfit_items = load_tok_jsonl(OVERFIT_JSONL, 16)
assert len(overfit_items) == 16, f'Expected 16 overfit items, got {len(overfit_items)}'

print(f'Overfit: {len(overfit_items)} items, {OVERFIT_STEPS} steps, lr={LR_OVERFIT}, '
      f'micro_batch={MICRO_BATCH_OVERFIT}')

# Reload model
model_ov = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR), torch_dtype=DTYPE, device_map={'':0},
    attn_implementation='sdpa', local_files_only=True,
)
model_ov.config.use_cache = False
model_ov = get_peft_model(model_ov, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.0,  # dropout=0 for overfit
    target_modules=target_modules, bias='none', task_type=TaskType.CAUSAL_LM
))
for n,p in model_ov.named_parameters():
    if 'lora_' in n: p.data = p.data.to(torch.float32)

collate_fn_ov = make_collator(tok.pad_token_id)
opt_ov = AdamW(model_ov.parameters(), lr=LR_OVERFIT, weight_decay=0.0)

initial_loss = None
loss_curve = []

model_ov.train()
for step in range(OVERFIT_STEPS):
    # Cycle through overfit items
    start_i = (step * MICRO_BATCH_OVERFIT) % len(overfit_items)
    batch_items = [overfit_items[(start_i + j) % len(overfit_items)]
                   for j in range(MICRO_BATCH_OVERFIT)]
    batch = collate_fn_ov(batch_items)
    batch = {k: v.to(device) if hasattr(v,'to') else v for k,v in batch.items()}
    outputs = model_ov(**{k:v for k,v in batch.items() if k!='num_items_in_batch'})
    loss = outputs.loss
    opt_ov.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model_ov.parameters(), 1.0)
    opt_ov.step()
    loss_val = loss.item()
    if step == 0:
        initial_loss = loss_val
    if step % 25 == 0 or step == OVERFIT_STEPS - 1:
        loss_curve.append({'step': step, 'loss': round(loss_val, 6)})

final_loss = loss_val
ratio = final_loss / max(initial_loss, 1e-9)
pass_criterion = ratio <= 0.30

overfit_report = {
    'initial_loss': round(initial_loss, 6),
    'final_loss': round(final_loss, 6),
    'ratio_final_to_initial': round(ratio, 4),
    'pass_30pct': pass_criterion,
    'steps': OVERFIT_STEPS,
    'loss_curve': loss_curve,
    'initial_defined_as': 'loss at step 0',
    'final_defined_as': f'loss at step {OVERFIT_STEPS-1}',
}
write_result('overfit_results', overfit_report)
print(f'Overfit: initial={initial_loss:.4f}, final={final_loss:.4f}, '
      f'ratio={ratio:.3f}, PASS={pass_criterion}')
del model_ov, opt_ov
gc.collect(); torch.cuda.empty_cache()

## Section: Kill-and-Resume test

In [ ]:
# Kill-and-resume test
# 1. Train to step K, save checkpoint with optimizer + RNG state
# 2. Record loss at step K+1 from uninterrupted run
# 3. Resume from checkpoint, record loss at step K+1
# 4. Assert |resumed - uninterrupted| / uninterrupted < 2%

from train_lora import TokenMeanLossTrainer
from transformers import TrainingArguments, Trainer, set_seed
import shutil

RESUME_K = 10   # checkpoint at step K
MICRO_BS_RESUME = e0_results['best_micro_batch']
GRAD_ACCUM_RESUME = max(1, 32 // MICRO_BS_RESUME)
LR_RESUME = 1e-4
set_seed(42)

print(f'Kill-resume: K={RESUME_K}, micro_batch={MICRO_BS_RESUME}, '
      f'grad_accum={GRAD_ACCUM_RESUME}')

overfit_ds = TokenizedDataset(overfit_items)

def make_trainer(out_subdir, n_steps, seed=42):
    set_seed(seed)
    model_r = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR), torch_dtype=DTYPE, device_map={'':0},
        attn_implementation='sdpa', local_files_only=True,
    )
    model_r.config.use_cache = False
    model_r = get_peft_model(model_r, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=target_modules, bias='none', task_type=TaskType.CAUSAL_LM
    ))
    for n,p in model_r.named_parameters():
        if 'lora_' in n: p.data = p.data.to(torch.float32)

    args = TrainingArguments(
        output_dir=str(KAGGLE_WORKING / out_subdir),
        max_steps=n_steps,
        per_device_train_batch_size=MICRO_BS_RESUME,
        gradient_accumulation_steps=GRAD_ACCUM_RESUME,
        learning_rate=LR_RESUME,
        lr_scheduler_type='cosine', warmup_ratio=0.03,
        weight_decay=0.0, max_grad_norm=1.0,
        bf16=(PRECISION=='bf16'), fp16=(PRECISION=='fp16'),
        save_strategy='steps', save_steps=RESUME_K,
        save_total_limit=2, logging_steps=1,
        seed=seed, data_seed=seed, push_to_hub=False,
    )

    class _Trainer(TokenMeanLossTrainer, Trainer): pass
    return _Trainer(
        model=model_r, args=args,
        train_dataset=overfit_ds,
        data_collator=make_collator(tok.pad_token_id),
    ), model_r

# Uninterrupted run: K+1 steps, record step K+1 loss
trainer_full, model_full = make_trainer('resume_full', RESUME_K + 1)
trainer_full.train()
log_full = [l for l in trainer_full.state.log_history if 'loss' in l and 'step' in l]
loss_at_k1_full = next((l['loss'] for l in log_full if l['step'] == RESUME_K + 1), None)
if loss_at_k1_full is None and log_full:
    loss_at_k1_full = log_full[-1]['loss']
del trainer_full, model_full
gc.collect(); torch.cuda.empty_cache()

# Checkpoint run: train to K, then resume to K+1
ckpt_dir = KAGGLE_WORKING / 'resume_ckpt'
trainer_ckpt, model_ckpt = make_trainer('resume_ckpt', RESUME_K)
trainer_ckpt.train()
saved_ckpts = sorted((ckpt_dir).glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
del trainer_ckpt, model_ckpt
gc.collect(); torch.cuda.empty_cache()

# Resume from checkpoint K
if saved_ckpts:
    trainer_res, model_res = make_trainer('resume_cont', RESUME_K + 1)
    trainer_res.train(resume_from_checkpoint=str(saved_ckpts[-1]))
    log_res = [l for l in trainer_res.state.log_history if 'loss' in l and 'step' in l]
    loss_at_k1_resumed = next((l['loss'] for l in log_res if l['step'] == RESUME_K + 1), None)
    if loss_at_k1_resumed is None and log_res:
        loss_at_k1_resumed = log_res[-1]['loss']
    del trainer_res, model_res
    gc.collect(); torch.cuda.empty_cache()
else:
    loss_at_k1_resumed = None

if loss_at_k1_full and loss_at_k1_resumed:
    rel_diff = abs(loss_at_k1_resumed - loss_at_k1_full) / max(abs(loss_at_k1_full), 1e-9)
    resume_pass = rel_diff < 0.02
else:
    rel_diff = None
    resume_pass = False

resume_report = {
    'k': RESUME_K,
    'loss_step_k1_uninterrupted': loss_at_k1_full,
    'loss_step_k1_resumed': loss_at_k1_resumed,
    'relative_diff': round(rel_diff, 6) if rel_diff is not None else None,
    'pass_2pct': resume_pass,
}
write_result('resume_results', resume_report)
print(f'Resume: uninterrupted={loss_at_k1_full}, resumed={loss_at_k1_resumed}, '
      f'rel_diff={rel_diff}, PASS={resume_pass}')

## Section: T7 — Gradient accumulation equivalence

In [ ]:
# T7: micro-batch 1 × accumulation 4 vs micro-batch 4 × accumulation 1
# fp32, LoRA dropout=0, same 4 dialogues, assert max gradient diff < 1e-3
set_seed(42)
FOUR_ITEMS = overfit_items[:4]

def load_model_fp32():
    model_t7 = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR), torch_dtype=torch.float32, device_map={'':0},
        attn_implementation='sdpa', local_files_only=True,
    )
    model_t7.config.use_cache = False
    model_t7 = get_peft_model(model_t7, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.0,
        target_modules=target_modules, bias='none', task_type=TaskType.CAUSAL_LM
    ))
    return model_t7

collate_t7 = make_collator(tok.pad_token_id)

# Run A: micro-batch 1, accumulate 4
model_a = load_model_fp32()
model_a.train()
model_a.zero_grad()
for item in FOUR_ITEMS:
    b = collate_t7([item])
    b = {k: v.to(device) if hasattr(v,'to') else v for k,v in b.items()}
    out = model_a(**{k:v for k,v in b.items() if k!='num_items_in_batch'})
    # scale loss by 1/4 to match gradient magnitude
    (out.loss / 4.0).backward()
grads_a = {n: p.grad.clone().detach() for n,p in model_a.named_parameters()
           if p.requires_grad and p.grad is not None}
del model_a; gc.collect(); torch.cuda.empty_cache()

# Run B: micro-batch 4, accumulate 1
model_b = load_model_fp32()
model_b.train()
model_b.zero_grad()
b4 = collate_t7(FOUR_ITEMS)
b4 = {k: v.to(device) if hasattr(v,'to') else v for k,v in b4.items()}
out4 = model_b(**{k:v for k,v in b4.items() if k!='num_items_in_batch'})
out4.loss.backward()
grads_b = {n: p.grad.clone().detach() for n,p in model_b.named_parameters()
           if p.requires_grad and p.grad is not None}
del model_b; gc.collect(); torch.cuda.empty_cache()

# Compare
max_abs_diff = 0.0
max_rel_diff = 0.0
for name in grads_a:
    if name in grads_b:
        diff = (grads_a[name] - grads_b[name]).abs()
        abs_max = diff.max().item()
        rel_max = (diff / (grads_b[name].abs() + 1e-9)).max().item()
        max_abs_diff = max(max_abs_diff, abs_max)
        max_rel_diff = max(max_rel_diff, rel_max)

t7_pass = max_abs_diff < 1e-3
t7_report = {
    'max_absolute_grad_diff': round(max_abs_diff, 8),
    'max_relative_grad_diff': round(max_rel_diff, 6),
    'threshold': 1e-3,
    'pass': t7_pass,
    'n_params_compared': len(grads_a),
}
write_result('t7_results', t7_report)
print(f'T7: max_abs_diff={max_abs_diff:.2e}, pass={t7_pass}')

## Section: E2 — 150-step LR pilots

In [ ]:
# E2: 3 × 150-step pilots at LR 5e-5 / 1e-4 / 2e-4
# Identical pilot mix and seed for all three
# Save val loss curve; pick lowest macro val loss with no spikes

E2_LRS = [5e-5, 1e-4, 2e-4]
E2_STEPS = 150
MICRO_BS_E2 = e0_results['best_micro_batch']
GRAD_ACCUM_E2 = max(1, 32 // MICRO_BS_E2)

print(f'E2: {len(E2_LRS)} pilots × {E2_STEPS} steps, micro_batch={MICRO_BS_E2}')

# Load pilot mix
pilot_all = load_tok_jsonl(PILOT_JSONL, 4800)
pilot_ds = TokenizedDataset(pilot_all)

# Load val sets
val_md_items = load_tok_jsonl(VAL_MD_JSONL, 200)
val_hc_items = load_tok_jsonl(VAL_HC_JSONL, 200)
val_md_ds = TokenizedDataset(val_md_items)
val_hc_ds = TokenizedDataset(val_hc_items)

e2_results = {}

for lr in E2_LRS:
    lr_tag = f'lr_{lr:.0e}'.replace('-0', 'e-')
    print(f'\nE2 lr={lr:.0e}...')
    set_seed(42)

    model_e2 = AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR), torch_dtype=DTYPE, device_map={'':0},
        attn_implementation='sdpa', local_files_only=True,
    )
    model_e2.config.use_cache = False
    model_e2 = get_peft_model(model_e2, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=target_modules, bias='none', task_type=TaskType.CAUSAL_LM
    ))
    for n,p in model_e2.named_parameters():
        if 'lora_' in n: p.data = p.data.to(torch.float32)

    out_e2 = KAGGLE_WORKING / f'e2_{lr_tag}'

    args_e2 = TrainingArguments(
        output_dir=str(out_e2),
        max_steps=E2_STEPS,
        per_device_train_batch_size=MICRO_BS_E2,
        gradient_accumulation_steps=GRAD_ACCUM_E2,
        learning_rate=lr,
        lr_scheduler_type='cosine', warmup_ratio=0.03,
        weight_decay=0.0, max_grad_norm=1.0,
        bf16=(PRECISION=='bf16'), fp16=(PRECISION=='fp16'),
        eval_strategy='steps', eval_steps=50,
        save_strategy='steps', save_steps=50,
        save_total_limit=2, logging_steps=10,
        seed=42, data_seed=42, push_to_hub=False,
        load_best_model_at_end=False,
    )

    class _E2Trainer(TokenMeanLossTrainer, Trainer):
        def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix='eval'):
            md_m = super().evaluate(eval_dataset=val_md_ds, metric_key_prefix='eval_md')
            hc_m = super().evaluate(eval_dataset=val_hc_ds, metric_key_prefix='eval_hc')
            md_loss = md_m.get('eval_md_loss', float('inf'))
            hc_loss = hc_m.get('eval_hc_loss', float('inf'))
            macro = (md_loss + hc_loss) / 2.0
            return {**md_m, **hc_m, 'eval_loss': macro, 'eval_macro': macro}

    trainer_e2 = _E2Trainer(
        model=model_e2, args=args_e2,
        train_dataset=pilot_ds,
        eval_dataset=val_md_ds,
        data_collator=make_collator(tok.pad_token_id),
    )
    trainer_e2.train()

    # Extract val loss curve (no corpus text — only numbers)
    val_curve = [
        {'step': l.get('step'), 'eval_macro': l.get('eval_macro'), 'eval_loss': l.get('eval_loss')}
        for l in trainer_e2.state.log_history
        if 'eval_loss' in l
    ]
    train_curve = [
        {'step': l.get('step'), 'loss': l.get('loss')}
        for l in trainer_e2.state.log_history
        if 'loss' in l and 'eval_loss' not in l
    ]
    final_macro = val_curve[-1]['eval_macro'] if val_curve else None

    e2_results[lr_tag] = {
        'lr': lr,
        'steps': E2_STEPS,
        'final_macro_val_loss': final_macro,
        'val_curve': val_curve,
        'train_curve': train_curve,
    }
    print(f'  E2 lr={lr:.0e}: final macro_val={final_macro}')

    del model_e2, trainer_e2
    gc.collect(); torch.cuda.empty_cache()

write_result('e2_results', e2_results)
print('E2 complete.')

## Final status

In [ ]:
# Final status cell
import json
from pathlib import Path

results_files = sorted(RESULTS_DIR.glob('*.json'))
print('=== P3 KAGGLE STATUS ===')
for f in results_files:
    size = f.stat().st_size
    sha = sha256_file(f)
    print(f'  {f.name:40s}  {size:8,} B  sha256={sha[:16]}...')

# Summary of pass/fail
summary = {}
for key, fname in [
    ('t1_t6', 't1_t6_results.json'),
    ('e0',    'e0_results.json'),
    ('overfit', 'overfit_results.json'),
    ('resume', 'resume_results.json'),
    ('t7',    't7_results.json'),
]:
    p = RESULTS_DIR / fname
    if p.exists():
        d = json.loads(p.read_text(encoding='utf-8'))
        summary[key] = d.get('pass', d.get('pass_30pct', d.get('pass_2pct', 'N/A')))

if (RESULTS_DIR / 'e2_results.json').exists():
    e2d = json.loads((RESULTS_DIR / 'e2_results.json').read_text(encoding='utf-8'))
    macros = {k: v.get('final_macro_val_loss') for k,v in e2d.items()}
    summary['e2_macros'] = macros

print('\nAcceptance summary:')
for k, v in summary.items():
    print(f'  {k}: {v}')

print('\n=== NOTEBOOK COMPLETE ===')
print('Download all files from /kaggle/working/results/ to sft/results/p3/')